<a href="https://colab.research.google.com/github/AmarSai01/nvidia-gpu-ml-jax-labs/blob/main/rapids-nyc-taxi/NYC_Taxi_GPU_Acceleration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GPU-Accelerated ML on NYC Taxi Data (NVIDIA RAPIDS)
Predicting taxi tip amounts from ~3M NYC trips using pandas, scikit-learn and XGBoost,
accelerated on an NVIDIA T4 GPU with cuDF and cuML (zero code changes).

**Results:** XGBoost training was ~4.6x faster on GPU (1.0s vs 4.6s on CPU).
Best model: Random Forest, R² = 0.583.

In [ ]:
!nvidia-smi

Wed Oct 28 21:05:27 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
%load_ext cudf.pandas
%load_ext cuml.accel
import pandas as pd, numpy as np, time

The cudf.pandas extension is already loaded. To reload it, use:
  %reload_ext cudf.pandas
The cuml.accel extension is already loaded. To reload it, use:
  %reload_ext cuml.accel


In [ ]:
!wget -q https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet -O taxi.parquet
df = pd.read_parquet("taxi.parquet")
print(df.shape)
df.head()

(2964624, 19)


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee
0,2,2024-01-01 00:57:55,2024-01-01 01:17:43,1,1.72,1,N,186,79,2,17.7,1.0,0.5,0.00,0.0,1.0,22.70,2.5,0.0
1,1,2024-01-01 00:03:00,2024-01-01 00:09:36,1,1.80,1,N,140,236,1,10.0,3.5,0.5,3.75,0.0,1.0,18.75,2.5,0.0
2,1,2024-01-01 00:17:06,2024-01-01 00:35:01,1,4.70,1,N,236,79,1,23.3,3.5,0.5,3.00,0.0,1.0,31.30,2.5,0.0
3,1,2024-01-01 00:36:38,2024-01-01 00:44:56,1,1.40,1,N,79,211,1,10.0,3.5,0.5,2.00,0.0,1.0,17.00,2.5,0.0
4,1,2024-01-01 00:46:51,2024-01-01 00:52:57,1,0.80,1,N,211,148,1,7.9,3.5,0.5,3.20,0.0,1.0,16.10,2.5,0.0


In [ ]:
df = df[(df.payment_type == 1) & (df.fare_amount > 0) & (df.trip_distance > 0)]
df["hour"] = df.tpep_pickup_datetime.dt.hour
df["weekday"] = df.tpep_pickup_datetime.dt.weekday
features = ["trip_distance", "fare_amount", "passenger_count", "hour", "weekday", "PULocationID", "DOLocationID"]
df = df.dropna(subset=features + ["tip_amount"])
X = df[features].astype("float32")
y = df["tip_amount"].astype("float32")
print(X.shape)

(2298390, 7)


In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb

Xs, ys = X.sample(500_000, random_state=42), y.sample(500_000, random_state=42)

for name, model in [
    ("Linear Regression", LinearRegression()),
    ("Random Forest", RandomForestRegressor(n_estimators=50, max_depth=10)),
    ("XGBoost (GPU)", xgb.XGBRegressor(n_estimators=200, max_depth=8, device="cuda")),
]:
    t = time.time()
    scores = cross_val_score(model, Xs, ys, cv=3, scoring="r2")
    print(f"{name}: R2={scores.mean():.3f}  time={time.time()-t:.1f}s")

Linear Regression: R2=0.540  time=1.0s


/usr/local/lib/python3.13/dist-packages/treelite/core.py:25: UserWarning: [21:08:36] /__w/nvforest/nvforest/python/nvforest/build/cp311-abi3-linux_x86_64/_deps/treelite-src/src/serializer.cc:202: The model you are loading originated from a newer Treelite version; some functionalities may be unavailable.
Currently running Treelite version 4.6.1
The model checkpoint was generated from Treelite version 4.8.0
  warnings.warn(py_str(msg))
/usr/local/lib/python3.13/dist-packages/treelite/core.py:25: UserWarning: [21:08:37] /__w/nvforest/nvforest/python/nvforest/build/cp311-abi3-linux_x86_64/_deps/treelite-src/src/serializer.cc:202: The model you are loading originated from a newer Treelite version; some functionalities may be unavailable.
Currently running Treelite version 4.6.1
The model checkpoint was generated from Treelite version 4.8.0
  warnings.warn(py_str(msg))
/usr/local/lib/python3.13/dist-packages/treelite/core.py:25: UserWarning: [21:08:38] /__w/nvforest/nvforest/python/nvforest/

Random Forest: R2=0.583  time=2.9s
XGBoost (GPU): R2=0.553  time=2.7s


In [ ]:
for device in ["cpu", "cuda"]:
    m = xgb.XGBRegressor(n_estimators=200, max_depth=8, device=device)
    t = time.time(); m.fit(Xs, ys)
    print(f"{device.upper()}: {time.time()-t:.1f}s")

CPU: 4.6s
CUDA: 1.0s


In [ ]:
%%cudf.pandas.profile
df.groupby("PULocationID")["tip_amount"].mean().sort_values(ascending=False).head(10)

,tip_amount
PULocationID,
44,58.750000
251,24.600000
172,21.810000
6,19.305000
8,15.737500
154,14.168235
265,13.365434
93,13.131258
207,12.966190


                                                                                                                  
                                            Total time elapsed: 1.410 seconds                                     
                                          7 GPU function calls in 0.395 seconds                                   
                                          2 CPU function calls in 0.015 seconds                                   
                                                                                                                  
                                                          Stats                                                   
                                                                                                                  
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━┓
┃ Function                     ┃ GPU ncalls ┃ GPU cumtime ┃ GPU percall ┃ CPU ncalls ┃ CPU cumtime ┃ CPU percall ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━┩
│ DataFrame.groupby            │ 1          │ 0.080       │ 0.080       │ 0          │ 0.000       │ 0.000       │
│ DataFrameGroupBy.__getitem__ │ 1          │ 0.005       │ 0.005       │ 0          │ 0.000       │ 0.000       │
│ GroupBy.mean                 │ 1          │ 0.280       │ 0.280       │ 0          │ 0.000       │ 0.000       │
│ Series.sort_values           │ 1          │ 0.021       │ 0.021       │ 0          │ 0.000       │ 0.000       │
│ NDFrame.head                 │ 1          │ 0.001       │ 0.001       │ 0          │ 0.000       │ 0.000       │
│ Series.__repr__              │ 1          │ 0.006       │ 0.006       │ 0          │ 0.000       │ 0.000       │
│ Series.to_frame              │ 1          │ 0.000       │ 0.000       │ 0          │ 0.000       │ 0.000       │
│ DataFrame._repr_html_        │ 0          │ 0.000       │ 0.000       │ 1          │ 0.011       │ 0.011       │
│ NDFrame._repr_latex_         │ 0          │ 0.000       │ 0.000       │ 1          │ 0.003       │ 0.003       │
└──────────────────────────────┴────────────┴─────────────┴─────────────┴────────────┴─────────────┴─────────────┘

Not all pandas operations ran on the GPU. The following functions required CPU fallback:

- DataFrame._repr_html_
- NDFrame._repr_latex_

To request GPU support for any of these functions, please file a Github issue here: 
]8;id=121730;https://github.com/rapidsai/cudf/issues/new?assignees=&labels=%3F+-+Needs+Triage%2C+feature+request&projects=&template=pandas_function_request.md&title=%5BFEA%5D\https://github.com/rapidsai/cudf/issues/new/choose]8;;\.